In [6]:
import aiohttp
import asyncio
import nest_asyncio
import os
from dotenv import load_dotenv
from typing import List, Dict, Any, Tuple, Union
import json
from github import Github, Auth
from gidgethub.aiohttp import GitHubAPI
from gidgethub import GitHubException
import sys
import base64
from stamina import retry, retry_context
import re
from functools import partial
from textacy import preprocessing
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from qdrant_client.http.models import Distance, VectorParams
import uuid
from searcharray import SearchArray
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langgraph.graph import START, StateGraph
from langchain_core.documents import Document
from typing_extensions import List, TypedDict
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from langchain_cohere import CohereRerank
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# allow for multiple event loops within Jupyter
nest_asyncio.apply()

# load environment variables
load_dotenv()

import pickle

from pydantic import ConfigDict
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from langchain_core.callbacks import CallbackManagerForRetrieverRun

In [4]:
MAX_CHARACTERS = 30_000
NAMESPACE = uuid.NAMESPACE_URL

def strip_markdown(text: str) -> str:
    """Remove common Markdown syntax; keep inner text and emojis."""
    # Headings: ## Title -> Title
    text = re.sub(r"^#{1,6}\s*", "", text, flags=re.MULTILINE)
    # Inline code: `code` -> code
    text = re.sub(r"`([^`]+)`", r"\1", text)
    # Bold/italic: **bold** or *italic* -> bold / italic
    text = re.sub(r"\*{1,2}([^*]+)\*{1,2}", r"\1", text)
    text = re.sub(r"_{1,2}([^_]+)_{1,2}", r"\1", text)
    # Link markup: [text](url) -> text
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)
    # Collapse many newlines
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text


def make_normalize_text_pipeline(
    *,
    url_repl: str = "",
    unicode_form: str = "NFC",
):
    """
    Build a textacy pipeline: Markdown/HTML → plain text (emojis kept).

    Args:
        url_repl: Replacement for URLs (default ""). Use "_URL_" to keep a placeholder.
        unicode_form: Unicode normalization form ("NFC", "NFD", "NFKC", "NFKD"). Default "NFC".

    Returns:
        A callable that takes a string and returns preprocessed plain text.
    """
    return preprocessing.make_pipeline(
        strip_markdown,
        preprocessing.remove.html_tags,
        preprocessing.normalize.bullet_points,
        preprocessing.normalize.quotation_marks,
        partial(preprocessing.normalize.unicode, form=unicode_form),
        preprocessing.normalize.whitespace,
    )


# Default pipeline instance
normalize_text_pipeline = make_normalize_text_pipeline()

def repo_to_uuid(repo_name: str) -> str:
    return str(uuid.uuid5(NAMESPACE, repo_name))

def normalize_docs(docs: List[Dict[str, Any]]):
    content_str = ''
    for doc in docs:
        content_str += (doc['content'] + '\n\n')
    truncated = content_str[:MAX_CHARACTERS] if len(content_str) > MAX_CHARACTERS else content_str
    normalized_text = normalize_text_pipeline(truncated)
    return normalized_text


def preprocess_text(text: str | None) -> List[str]:
    """borrowed from Ask Brave search for common preprocessing/tokenizer steps before TF-IDF"""
    if text is None:
        return []
    # Lowercase
    text = text.lower()
    # Remove punctuation and digits
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords and short words
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]
    # Lemmatize
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return tokens

In [3]:
# load cached github data
with open("../data/cached/github_data.pkl", "rb") as f:
    starred_repo_data = pickle.load(f)

In [9]:
# step 2: build the per-field DataFrame
search_df = pd.DataFrame([
    {
        "id":          repo_to_uuid(repo["repo"]),
        "repo":        repo["repo"],
        "description": repo.get("description") or "",
        "topics":      " ".join(repo.get("topics", [])),
        "content":     f"Topics: {','.join(repo.get('topics', []))}\n" + normalize_docs(repo["docs"]),
    }
    for repo in starred_repo_data
])

search_df["repo_idx"]        = SearchArray.index(search_df["repo"],        tokenizer=preprocess_text)
search_df["description_idx"] = SearchArray.index(search_df["description"], tokenizer=preprocess_text)
search_df["topics_idx"]      = SearchArray.index(search_df["topics"],      tokenizer=preprocess_text)
search_df["content_idx"]     = SearchArray.index(search_df["content"],     tokenizer=preprocess_text)

2026-03-14 17:34:16,693 - searcharray.indexing - INFO - Indexing begins w/ 4 workers
2026-03-14 17:34:16,694 - searcharray.indexing - INFO - 0 Batch Start tokenization
2026-03-14 17:34:16,694 - searcharray.indexing - INFO - Tokenizing 2049 documents
2026-03-14 17:34:18,375 - searcharray.indexing - INFO - Tokenization -- vstacking
2026-03-14 17:34:18,377 - searcharray.indexing - INFO - Tokenization -- DONE
2026-03-14 17:34:18,378 - searcharray.indexing - INFO - Inverting docs->terms
2026-03-14 17:34:18,380 - searcharray.indexing - INFO - Encoding positions to bit array
2026-03-14 17:34:18,382 - searcharray.indexing - INFO - Batch tokenization complete
2026-03-14 17:34:18,383 - searcharray.indexing - INFO - (main thread) Processing 1 batch results
2026-03-14 17:34:18,383 - searcharray.indexing - INFO - Indexing from tokenization complete
2026-03-14 17:34:18,385 - searcharray.indexing - INFO - Indexing begins w/ 4 workers
2026-03-14 17:34:18,386 - searcharray.indexing - INFO - 0 Batch Sta

In [10]:
# step 3: define the multi-match-search function (that we came up with in the labeling notebook); it allows you to boost the BM25 score for certain fields
def multi_match_search(
    query: str,
    df: pd.DataFrame,
    columns: list[str],
    boosts: dict[str, float] | None = None,
) -> pd.DataFrame:
    if boosts is None:
        boosts = {}
    boost_values = {col: boosts.get(col, 1.0) for col in columns}

    # Tokenize query using each field's own tokenizer
    tokenized_queries = {
        col: df[col].array.tokenizer(query)
        for col in columns
    }

    # Score each field per query term, apply boost → shape (num_terms x num_docs)
    field_scores = {
        col: np.asarray([df[col].array.score(term) for term in tokenized_queries[col]]) * boost_values[col]
        for col in columns
    }

    num_terms = max(len(scores) for scores in field_scores.values())

    # Dismax: for each term, take the best score across all fields
    best_term_scores_per_doc = []
    for term_idx in range(num_terms):
        term_scores_across_fields = [
            field_scores[col][term_idx]
            for col in columns
            if term_idx < len(field_scores[col])
        ]
        best_score = np.max(term_scores_across_fields, axis=0)
        best_term_scores_per_doc.append(best_score)

    result = df.copy()
    result["score"] = np.sum(best_term_scores_per_doc, axis=0)
    return result.sort_values("score", ascending=False)

In [11]:
# step 4: create MultiMatchBM25Retriever by subclassing LangChain's BaseRetriever
class MultiMatchBM25Retriever(BaseRetriever):
    """
    LangChain-compatible retriever wrapping SearchArray multi-field BM25
    with per-field boost weights and preprocess_text tokenization.
    """
    # Pydantic v2 replaces `class Config` with `model_config`
    model_config = ConfigDict(arbitrary_types_allowed=True)

    search_df: pd.DataFrame
    columns: list[str]
    boosts: dict[str, float]
    k: int = 10

    def _get_relevant_documents(
        self,
        query: str,
        *,
        run_manager: CallbackManagerForRetrieverRun,
    ) -> list[Document]:
        results = multi_match_search(
            query=query,
            df=self.search_df,
            columns=self.columns,
            boosts=self.boosts,
        )
        top_k = results[results["score"] > 0].head(self.k)

        return [
            Document(
                page_content=row["content"],
                metadata={
                    "id":          row["id"],
                    "repo":        row["repo"],
                    "description": row["description"],
                    "topics":      row["topics"],
                    "score":       float(row["score"]),
                }
            )
            for _, row in top_k.iterrows()
        ]

In [16]:
# compute BM25 curated list score for each document (git repo)
CURATED_QUERY = "awesome curated lists"

# Run multi_match_search for the curated query across your indexed search_df
curated_results = multi_match_search(
    query=CURATED_QUERY,
    df=search_df,
    columns=["repo_idx", "topics_idx", "description_idx", "content_idx"],
    boosts={
        "repo_idx":         3.0,
        "topics_idx":       2.0,
        "description_idx":  1.5,
        "content_idx":      1.0,
    },
)

# Build a lookup dict: id → curated_list_bm25_score
curated_score_by_id = dict(
    zip(curated_results["id"], curated_results["score"].astype(float))
)

# add curated list score to each document's metadata
documents = [
    Document(
        page_content=f"Topics: {','.join(repo.get('topics', []))}\n" + normalize_docs(repo["docs"]),
        metadata={
            "id":                (id_ := repo_to_uuid(repo["repo"])),
            "repo":              repo["repo"],
            "description":       repo.get("description") or "",
            "topics":            repo.get("topics", []),
            "language":          repo.get("language"),
            "doc_source":        repo.get("doc_source"),
            "stars":             repo.get("stars"),
            "url":               repo.get("url"),
            "curated_list_bm25": curated_score_by_id.get(id_, 0.0),
        }
    )
    for repo in starred_repo_data
]

In [18]:
# step 5: wire into Qdrant ensemble retriever
# ── Multi-match BM25 retriever ─────────────────────────────────────────────────
multi_match_retriever = MultiMatchBM25Retriever(
    search_df=search_df,
    columns=["repo_idx", "topics_idx", "description_idx", "content_idx"],
    boosts={
        "repo_idx":         3.0,
        "topics_idx":       2.0,
        "description_idx":  1.5,
        "content_idx":      1.0,
    },
    k=10,
)

# ── Qdrant dense retriever ─────────────────────────────────────────────────────
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
client = QdrantClient(":memory:")
client.create_collection(
    collection_name="ask_my_bookmark",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)
vector_store = QdrantVectorStore(
    client=client,
    collection_name="ask_my_bookmark",
    embedding=embeddings,
)
vector_store.add_documents(documents)
dense_retriever = vector_store.as_retriever(search_kwargs={"k": 10})

# ── Ensemble: multi-match BM25 + dense Qdrant (weighted RRF under the hood) ────────────
ensemble_retriever = EnsembleRetriever(
    retrievers=[multi_match_retriever, dense_retriever]
)

In [19]:
## test queries and see what is retrieved
query1 = "Do I have any starred repositories related to Bayesian Statistics or Bayesian Modeling?"

bm25_results  = multi_match_retriever.invoke(query1)
dense_results = dense_retriever.invoke(query1)
fused_results = ensemble_retriever.invoke(query1)

bm25_repos  = [doc.metadata["repo"] for doc in bm25_results]
dense_repos = [doc.metadata["repo"] for doc in dense_results]

print(f"{'Rank':<6} {'Repo':<45} {'In BM25?':<12} {'In Dense?'}")
print("-" * 85)
for i, doc in enumerate(fused_results):
    repo = doc.metadata["repo"]
    in_bm25  = "yes (#" + str(bm25_repos.index(repo) + 1)  + ")" if repo in bm25_repos  else "no"
    in_dense = "yes (#" + str(dense_repos.index(repo) + 1) + ")" if repo in dense_repos else "no"
    print(f"{i+1:<6} {repo:<45} {in_bm25:<12} {in_dense}")

Rank   Repo                                          In BM25?     In Dense?
-------------------------------------------------------------------------------------
1      pymc-devs/pymc-resources                      yes (#3)     yes (#2)
2      hyosubkim/bayes-toolbox                       yes (#5)     yes (#3)
3      bambinos/bambi                                yes (#2)     yes (#9)
4      avehtari/BDA_course_Aalto                     yes (#8)     yes (#5)
5      PacktPublishing/Bayesian-Analysis-with-Python-Second-Edition yes (#10)    yes (#6)
6      cognica-io/bayesian-bm25                      yes (#1)     no
7      kidaufo/StatisticalModeling                   no           yes (#1)
8      CamDavidsonPilon/Probabilistic-Programming-and-Bayesian-Methods-for-Hackers yes (#4)     no
9      BayesianModelingandComputationInPython/BookCode_Edition1 no           yes (#4)
10     bayesian-optimization/BayesianOptimization    yes (#6)     no
11     pymc-devs/pymc                             

In [21]:
len(fused_results)

15

In [22]:
len(dense_results)

10

In [23]:
len(bm25_results)

10

In [24]:
# build BM25 retriever
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 10

In [29]:
def compare_retrievers(query: str) -> None:
    multi_match_results = multi_match_retriever.invoke(query)
    legacy_bm25_results = bm25_retriever.invoke(query)
    dense_results       = dense_retriever.invoke(query)
    fused_results       = ensemble_retriever.invoke(query)

    multi_match_repos = [doc.metadata["repo"] for doc in multi_match_results]
    legacy_bm25_repos = [doc.metadata["repo"] for doc in legacy_bm25_results]
    dense_repos       = [doc.metadata["repo"] for doc in dense_results]
    fused_repos       = [doc.metadata["repo"] for doc in fused_results]

    def rank_label(repo, repo_list):
        return f"yes (#{repo_list.index(repo) + 1})" if repo in repo_list else "no"

    all_repos = list(dict.fromkeys(
        fused_repos + multi_match_repos + legacy_bm25_repos + dense_repos
    ))

    print(f"{'Rank':<6} {'Repo':<45} {'Fused?':<12} {'MultiMatch?':<14} {'LegacyBM25?':<14} {'Dense?'}")
    print("-" * 110)
    for i, repo in enumerate(all_repos):
        in_fused       = rank_label(repo, fused_repos)
        in_multi       = rank_label(repo, multi_match_repos)
        in_legacy_bm25 = rank_label(repo, legacy_bm25_repos)
        in_dense       = rank_label(repo, dense_repos)
        print(f"{i+1:<6} {repo:<45} {in_fused:<12} {in_multi:<14} {in_legacy_bm25:<14} {in_dense}")


compare_retrievers("Do I have any starred repositories related to Bayesian Statistics or Bayesian Modeling?")

Rank   Repo                                          Fused?       MultiMatch?    LegacyBM25?    Dense?
--------------------------------------------------------------------------------------------------------------
1      pymc-devs/pymc-resources                      yes (#1)     yes (#3)       no             yes (#2)
2      hyosubkim/bayes-toolbox                       yes (#2)     yes (#5)       no             yes (#3)
3      bambinos/bambi                                yes (#3)     yes (#2)       no             yes (#9)
4      avehtari/BDA_course_Aalto                     yes (#4)     yes (#8)       no             yes (#5)
5      PacktPublishing/Bayesian-Analysis-with-Python-Second-Edition yes (#5)     yes (#10)      yes (#7)       yes (#6)
6      cognica-io/bayesian-bm25                      yes (#6)     yes (#1)       no             no
7      kidaufo/StatisticalModeling                   yes (#7)     no             no             yes (#1)
8      CamDavidsonPilon/Probabilistic-Prog

In [27]:
compare_retrievers("What are some repos related to recommender systems that I have starred?")

Rank   Repo                                          Fused?       MultiMatch?    LegacyBM25?    Dense?
--------------------------------------------------------------------------------------------------------------
1      grahamjenson/list_of_recommender_systems      yes (#1)     yes (#1)       yes (#1)       yes (#1)
2      caserec/Datasets-for-Recommender-Systems      yes (#2)     yes (#2)       no             yes (#2)
3      tensorflow/recommenders                       yes (#3)     yes (#5)       no             yes (#10)
4      robi56/Deep-Learning-for-Recommendation-Systems yes (#4)     yes (#9)       no             yes (#6)
5      massquantity/LibRecommender                   yes (#5)     yes (#8)       no             yes (#7)
6      LincolnBurrows2017/gauss-awesome-recommender-system-engine yes (#6)     yes (#3)       no             no
7      chihming/competitive-recsys                   yes (#7)     no             no             yes (#3)
8      NVIDIA-Merlin/systems             

In [28]:
compare_retrievers("What are the top recommender systems libraries that I have starred in the past?")

Rank   Repo                                          Fused?       MultiMatch?    LegacyBM25?    Dense?
--------------------------------------------------------------------------------------------------------------
1      grahamjenson/list_of_recommender_systems      yes (#1)     yes (#1)       yes (#1)       yes (#1)
2      caserec/Datasets-for-Recommender-Systems      yes (#2)     yes (#3)       no             yes (#7)
3      robi56/Deep-Learning-for-Recommendation-Systems yes (#3)     yes (#9)       no             yes (#4)
4      LincolnBurrows2017/gauss-awesome-recommender-system-engine yes (#4)     yes (#5)       no             yes (#10)
5      caserec/CaseRecommender                       yes (#5)     yes (#2)       no             no
6      jihoo-kim/awesome-RecSys                      yes (#6)     no             no             yes (#2)
7      recommenders-team/recommenders                yes (#7)     no             no             yes (#3)
8      RecList/reclist                   

In [31]:
compare_retrievers("Do I have any starred repos related to Markov Chains?")

Rank   Repo                                          Fused?       MultiMatch?    LegacyBM25?    Dense?
--------------------------------------------------------------------------------------------------------------
1      chihming/competitive-recsys                   yes (#1)     yes (#5)       no             yes (#3)
2      maciejkula/spotlight                          yes (#2)     yes (#8)       no             yes (#2)
3      mxgmn/MarkovJunior                            yes (#3)     yes (#4)       yes (#2)       yes (#9)
4      pymc-devs/pymc                                yes (#4)     yes (#10)      no             yes (#7)
5      probml/dynamax                                yes (#5)     yes (#1)       no             no
6      probml/pml2-book                              yes (#6)     no             no             yes (#1)
7      costezki/awesome-nlprojects                   yes (#7)     yes (#2)       no             no
8      palantir/pyspark-style-guide                  yes (#8)  